## Panel building script

### Imports

In [1]:
import pandas as pd
import numpy as np

### Paths

In [2]:
# From 02_structural_models_panel_build.ipynb
QEDS_PATH = "data/processed/WB_QEDS/QEDS_SDDS.csv"
MSCI_PATH = "data/processed/MSCI_indices/mscicountryindex.csv"
OVX_PATH  = "data/processed/Macroeconomic_variables/OVXCLS.csv"
CREDIT_RATINGS_PATH = "data/processed/"

# From 03_baseline_cca_model(1).ipynb
CDS_PATH = "data/processed/CDS/Weekly_CDS.csv"
RF_PATH  = "data/processed/Macroeconomic_variables/DGS1.csv"

# Outputs
PANEL_OUT = "data/processed/structural_model_panel_built.csv"

### QEDS data and default barrier

In [3]:
DEBT_INDICATORS = {
    'debt_st': 'Gross Ext. Debt Pos., General Government, Short-term, All instruments, USD',
    'debt_lt': 'Gross Ext. Debt Pos., General Government, Long-term, All instruments, USD',
    'debt_total': 'Gross Ext. Debt Pos., General Government, All maturities, All instruments, USD',
}

# Default barrier: ST + 0.5 * LT
print("Loading QEDS...")
qeds = pd.read_csv(QEDS_PATH, delimiter=";")

quarter_cols = [c for c in qeds.columns
                if c not in ['Cleaned_Name', 'Indicator Name']
                and len(c) == 6 and 'Q' in c]

print(f"Found {len(quarter_cols)} quarters: {quarter_cols[0]} -> {quarter_cols[-1]}")

qeds_long = qeds.melt(
    id_vars=['Cleaned_Name', 'Indicator Name'],
    value_vars=quarter_cols,
    var_name='quarter',
    value_name='value'
)

qeds_long.columns = ['country', 'indicator', 'quarter', 'value']

qeds_long['date'] = pd.to_datetime(
    qeds_long['quarter'].str.replace('Q1', '-03-31')
                        .str.replace('Q2', '-06-30')
                        .str.replace('Q3', '-09-30')
                        .str.replace('Q4', '-12-31')
)

qeds_long['value'] = pd.to_numeric(qeds_long['value'], errors='coerce')
qeds_long = qeds_long.dropna(subset=['date'])

print(f"QEDS long: {len(qeds_long)} rows, {qeds_long['country'].nunique()} countries")

available = qeds_long['indicator'].unique()

matched = {}
for key, pattern in DEBT_INDICATORS.items():
    if pattern in available:
        matched[key] = pattern
    else:
        matches = [ind for ind in available if pattern.lower() in ind.lower()]
        if matches:
            matched[key] = matches[0]
            print(f"WARNING partial match for {key}: {matches[0]}")

print("Matched indicators:", matched)

if len(matched) == 0:
    raise ValueError("No debt indicators matched in QEDS.")

debt_data = qeds_long[qeds_long['indicator'].isin(matched.values())].copy()
inv_matched = {v: k for k, v in matched.items()}
debt_data['var'] = debt_data['indicator'].map(inv_matched)

debt_wide = (debt_data
             .pivot_table(index=['country', 'date'], columns='var', values='value', aggfunc='first')
             .reset_index())

# Compute default barrier (ST + 0.5 LT)
if 'debt_st' in debt_wide.columns and 'debt_lt' in debt_wide.columns:
    debt_wide['default_barrier'] = debt_wide['debt_st'].fillna(0) + 0.5 * debt_wide['debt_lt'].fillna(0)
else:
    raise ValueError("Missing debt_st or debt_lt after pivot; cannot compute default_barrier.")

print("Debt quarterly wide:", debt_wide.shape)
debt_wide.head()

Loading QEDS...
Found 110 quarters: 1998Q1 -> 2025Q2
QEDS long: 25542000 rows, 129 countries
Matched indicators: {'debt_st': 'Gross Ext. Debt Pos., General Government, Short-term, All instruments, USD', 'debt_lt': 'Gross Ext. Debt Pos., General Government, Long-term, All instruments, USD', 'debt_total': 'Gross Ext. Debt Pos., General Government, All maturities, All instruments, USD'}
Debt quarterly wide: (8931, 6)


var,country,date,debt_lt,debt_st,debt_total,default_barrier
0,Afghanistan,2017-03-31,2.120923e+09,0.0,2.120923e+09,1.060461e+09
1,Afghanistan,2017-06-30,2.110604e+09,0.0,2.110604e+09,1.055302e+09
2,Afghanistan,2017-09-30,2.186717e+09,0.0,2.186717e+09,1.093359e+09
3,Afghanistan,2017-12-31,2.168967e+09,0.0,2.168967e+09,1.084484e+09
4,Afghanistan,2018-03-31,2.193132e+09,0.0,2.193132e+09,1.096566e+09


### Load MSCI indices

In [4]:
print("Loading MSCI daily...")
msci_raw = pd.read_csv(MSCI_PATH, delimiter=",")

date_col = msci_raw.columns[0]
msci_raw['date'] = pd.to_datetime(msci_raw[date_col], format='%d.%m.%Y', errors='coerce')

if msci_raw['date'].isna().all():
    msci_raw['date'] = pd.to_datetime(msci_raw[date_col], dayfirst=True, errors='coerce')

msci_raw = msci_raw.drop(columns=[date_col])

country_cols = [c for c in msci_raw.columns if c != 'date']
print(f"MSCI countries found: {len(country_cols)}")

msci_daily = msci_raw.melt(
    id_vars=['date'],
    value_vars=country_cols,
    var_name='country',
    value_name='msci_index'
)

msci_daily['msci_index'] = pd.to_numeric(msci_daily['msci_index'], errors='coerce')
msci_daily = msci_daily.dropna(subset=['date', 'msci_index'])

print("MSCI daily:", msci_daily.shape, "| range:", msci_daily['date'].min(), "->", msci_daily['date'].max())

# Assign week-ending Friday (same logic as nb02)
msci_daily = msci_daily.sort_values(['country', 'date']).reset_index(drop=True)

msci_daily['week'] = msci_daily['date'].apply(
    lambda d: d + pd.Timedelta(days=(4 - d.dayofweek) % 7) if d.dayofweek <= 4
              else d + pd.Timedelta(days=(4 - d.dayofweek + 7))
)

# Daily log returns
msci_daily['log_ret'] = msci_daily.groupby('country')['msci_index'].transform(
    lambda x: np.log(x / x.shift(1))
)

# Weekly aggregation
msci_weekly = msci_daily.groupby(['country', 'week']).agg(
    msci_index=('msci_index', 'last'),
    msci_ret_weekly=('log_ret', 'sum'),
    n_obs=('log_ret', 'count')
).reset_index()

msci_weekly.columns = ['country', 'date', 'msci_index', 'msci_ret_weekly', 'n_obs']

# Rolling 52-week annualized volatility
msci_weekly = msci_weekly.sort_values(['country', 'date']).reset_index(drop=True)
msci_weekly['msci_vol_annual'] = msci_weekly.groupby('country')['msci_ret_weekly'].transform(
    lambda x: x.rolling(window=52, min_periods=12).std() * np.sqrt(52)
)

print("MSCI weekly:", msci_weekly.shape)
msci_weekly.tail()


Loading MSCI daily...
MSCI countries found: 84
MSCI daily: (464660, 3) | range: 1999-12-31 00:00:00 -> 2024-12-31 00:00:00
MSCI weekly: (93021, 6)


c:\Users\JuanFranciscoPerez\Projects\commodities_and_sovereigns\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


,country,date,msci_index,msci_ret_weekly,n_obs,msci_vol_annual
93016,Zhonghua,2024-12-06,1093.254,0.023906,5,0.288095
93017,Zhonghua,2024-12-13,1097.409,0.003793,5,0.287356
93018,Zhonghua,2024-12-20,1082.873,-0.013334,5,0.285083
93019,Zhonghua,2024-12-27,1097.485,0.013404,5,0.282161
93020,Zhonghua,2025-01-03,1091.423,-0.005539,2,0.280425


### Expand quaterly debt to weekly

In [5]:
print("Expanding quarterly debt to weekly (forward-fill)...")

weekly_dates = msci_weekly[['date']].drop_duplicates().sort_values('date').reset_index(drop=True)

debt_weekly_list = []

for c in debt_wide['country'].dropna().unique():
    cq = debt_wide[debt_wide['country'] == c].sort_values('date').copy()
    if len(cq) == 0:
        continue

    min_d, max_d = cq['date'].min(), cq['date'].max()
    w = weekly_dates[(weekly_dates['date'] >= min_d) & (weekly_dates['date'] <= max_d)].copy()
    if len(w) == 0:
        continue

    tmp = w.merge(cq, on='date', how='left').sort_values('date')
    tmp['country'] = c

    # forward fill debt variables
    for col in ['debt_st', 'debt_lt', 'debt_total', 'default_barrier']:
        if col in tmp.columns:
            tmp[col] = tmp[col].ffill()

    debt_weekly_list.append(tmp)

debt_weekly = pd.concat(debt_weekly_list, ignore_index=True)
print("Debt weekly:", debt_weekly.shape)
debt_weekly.head()


Expanding quarterly debt to weekly (forward-fill)...
Debt weekly: (111981, 6)


,date,country,debt_lt,debt_st,debt_total,default_barrier
0,2017-03-31,Afghanistan,2.120923e+09,0.0,2.120923e+09,1.060461e+09
1,2017-04-07,Afghanistan,2.120923e+09,0.0,2.120923e+09,1.060461e+09
2,2017-04-14,Afghanistan,2.120923e+09,0.0,2.120923e+09,1.060461e+09
3,2017-04-21,Afghanistan,2.120923e+09,0.0,2.120923e+09,1.060461e+09
4,2017-04-28,Afghanistan,2.120923e+09,0.0,2.120923e+09,1.060461e+09


# Merge debt + MSCI weekly

In [6]:
print("Merging weekly debt + weekly MSCI...")
panel = msci_weekly.merge(
    debt_weekly,
    on=['country', 'date'],
    how='inner'
)

print("Panel after debt+MSCI merge:", panel.shape)
panel.head()


Merging weekly debt + weekly MSCI...
Panel after debt+MSCI merge: (61943, 10)


,country,date,msci_index,msci_ret_weekly,n_obs,msci_vol_annual,debt_lt,debt_st,debt_total,default_barrier
0,Argentina,2010-12-03,1034.304,0.033729,3,NaN,5.237300e+10,5.181000e+09,5.755400e+10,3.136750e+10
1,Argentina,2010-12-10,1063.641,0.027969,5,NaN,5.237300e+10,5.181000e+09,5.755400e+10,3.136750e+10
2,Argentina,2010-12-17,1021.676,-0.040254,5,NaN,5.237300e+10,5.181000e+09,5.755400e+10,3.136750e+10
3,Argentina,2010-12-24,1029.518,0.007646,5,NaN,5.237300e+10,5.181000e+09,5.755400e+10,3.136750e+10
4,Argentina,2010-12-31,1044.573,0.014517,5,NaN,6.204517e+10,7.443373e+09,6.948855e+10,3.846596e+10


### Add OVX

In [7]:
# --- OVX: load and clean ---
ovx = pd.read_csv(OVX_PATH)
ovx["date"] = pd.to_datetime(ovx["Date"])
ovx["OVX"]  = pd.to_numeric(ovx["OVXCLS"], errors="coerce")
ovx = ovx[["date", "OVX"]].dropna().sort_values("date")

# --- make weekly Friday series (ffill so each Friday has last known value) ---
ovx_weekly = (
    ovx.set_index("date")
       .resample("W-FRI")
       .ffill()
)

# --- critical step: reindex OVX to panel dates and fill missing ---
panel_dates = pd.DatetimeIndex(panel["date"].sort_values().unique())

ovx_weekly = ovx_weekly.reindex(panel_dates).ffill().bfill()

ovx_weekly = ovx_weekly.reset_index().rename(columns={"index": "date"})

# --- merge ---
panel = panel.merge(ovx_weekly, on="date", how="left")

# sanity check
print("OVX NaNs after merge:", panel["OVX"].isna().sum())
print("Panel date range:", panel["date"].min(), "->", panel["date"].max())
print("OVX date range:", ovx["date"].min(), "->", ovx["date"].max())


OVX NaNs after merge: 0
Panel date range: 1999-12-31 00:00:00 -> 2025-01-03 00:00:00
OVX date range: 2007-05-10 00:00:00 -> 2026-01-28 00:00:00


### Add CDS|

In [8]:
print("Loading CDS weekly...")
cds_raw = pd.read_csv(CDS_PATH)
date_col = cds_raw.columns[0]

cds_raw['date'] = pd.to_datetime(cds_raw[date_col], errors='coerce')
if cds_raw['date'].isna().sum() > len(cds_raw) * 0.5:
    cds_raw['date'] = pd.to_datetime(cds_raw[date_col], dayfirst=True, errors='coerce')

country_cols = [c for c in cds_raw.columns if c not in [date_col, 'date']]

cds = cds_raw.melt(
    id_vars=['date'],
    value_vars=country_cols,
    var_name='country',
    value_name='cds_spread'
)
cds['cds_spread'] = pd.to_numeric(cds['cds_spread'], errors='coerce')
cds = cds.dropna(subset=['date', 'cds_spread'])

# standardize for merge (same as nb03)
panel['country_clean'] = panel['country'].str.strip().str.lower()
cds['country_clean'] = cds['country'].str.strip().str.lower()

panel = panel.merge(
    cds[['country_clean', 'date', 'cds_spread']],
    on=['country_clean', 'date'],
    how='inner'
)

print("Panel after CDS merge:", panel.shape, "| countries:", panel['country'].nunique())


Loading CDS weekly...
Panel after CDS merge: (42329, 13) | countries: 53


### Load risk-free rate

In [9]:
print("Loading RF (DGS1) and resampling weekly...")
rf_raw = pd.read_csv(RF_PATH)
rf_raw['date'] = pd.to_datetime(rf_raw['Date'], errors='coerce')
rf_raw['rf'] = pd.to_numeric(rf_raw['DGS1'], errors='coerce') / 100

rf = rf_raw[['date', 'rf']].dropna()
rf_weekly = rf.set_index('date').resample('W-FRI').last().reset_index()

panel = panel.merge(rf_weekly, on='date', how='left')
panel['rf'] = panel['rf'].ffill().fillna(0.02)

print("Panel after RF merge:", panel.shape)


Loading RF (DGS1) and resampling weekly...
Panel after RF merge: (42329, 14)


## Add average jump size and st dev

In [14]:
jump_data = pd.read_csv('data/processed/msci_jump_parameters.csv')
panel = panel.merge(jump_data[['country','lambda','theta','delta']], on='country')

### Clean and drop

In [15]:
# drop helper column if you want later; keep for now for merges/filters
print("Missing values (top):")
print(panel.isnull().sum().sort_values(ascending=False).head(20))

panel = panel.dropna(subset=['msci_vol_annual', 'default_barrier', 'cds_spread'])
print("Panel after dropna:", panel.shape)


Missing values (top):
debt_st            338
debt_lt            338
debt_total         338
default_barrier    338
country              0
n_obs                0
msci_ret_weekly      0
msci_index           0
date                 0
msci_vol_annual      0
OVX                  0
country_clean        0
cds_spread           0
rf                   0
lambda               0
theta                0
delta                0
dtype: int64
Panel after dropna: (15680, 17)


## Filter MSCI EM list

In [16]:
msci_em = [
    'brazil', 'chile', 'china', 'colombia', 'czech republic', 'czechia',
    'egypt', 'greece', 'hungary', 'india', 'indonesia', 'korea', 'south korea',
    'kuwait', 'malaysia', 'mexico', 'peru', 'philippines', 'poland', 'qatar',
    'saudi arabia', 'south africa', 'taiwan', 'thailand', 'turkey',
    'united arab emirates', 'uae'
]

panel['country_lower'] = panel['country'].str.strip().str.lower()
panel = panel[panel['country_lower'].isin(msci_em)].copy()

print("Panel after EM filter:", panel.shape, "| countries:", panel['country'].nunique())


Panel after EM filter: (15680, 18) | countries: 20


In [17]:
msci_em = [
    'brazil', 'chile', 'china', 'colombia', 'czech republic', 'czechia',
    'egypt', 'greece', 'hungary', 'india', 'indonesia', 'korea', 'south korea',
    'kuwait', 'malaysia', 'mexico', 'peru', 'philippines', 'poland', 'qatar',
    'saudi arabia', 'south africa', 'taiwan', 'thailand', 'turkey',
    'united arab emirates', 'uae'
]

panel['country_lower'] = panel['country'].str.strip().str.lower()
panel = panel[panel['country_lower'].isin(msci_em)].copy()

# ---- Date filter (EDIT THESE DATES AS NEEDED) ----
START_DATE = "2014-01-01"
END_DATE   = "2024-12-31"

panel = panel[(panel['date'] >= START_DATE) & (panel['date'] <= END_DATE)].copy()

print("Panel after EM + date filter:", panel.shape, "| countries:", panel['country'].nunique())
print("Date range:", panel['date'].min(), "->", panel['date'].max())


Panel after EM + date filter: (10961, 18) | countries: 20
Date range: 2014-01-03 00:00:00 -> 2024-12-27 00:00:00


In [18]:

panel.to_csv(PANEL_OUT, index=False)
print("Saved:", PANEL_OUT)

panel.head()

Saved: data/processed/structural_model_panel_built.csv


,country,date,msci_index,msci_ret_weekly,n_obs,msci_vol_annual,debt_lt,debt_st,debt_total,default_barrier,OVX,country_clean,cds_spread,rf,lambda,theta,delta,country_lower
315,Brazil,2014-01-03,804.695,-0.023366,5,0.220537,5.867838e+10,0.0,5.867838e+10,2.933919e+10,20.60,brazil,187.25,0.0013,0.10486,-0.024711,0.058012,brazil
316,Brazil,2014-01-10,788.350,-0.020521,5,0.220925,5.867838e+10,0.0,5.867838e+10,2.933919e+10,19.47,brazil,191.83,0.0012,0.10486,-0.024711,0.058012,brazil
317,Brazil,2014-01-17,780.773,-0.009658,5,0.220657,5.867838e+10,0.0,5.867838e+10,2.933919e+10,17.12,brazil,194.64,0.0011,0.10486,-0.024711,0.058012,brazil
318,Brazil,2014-01-24,743.312,-0.049169,5,0.224817,5.867838e+10,0.0,5.867838e+10,2.933919e+10,19.46,brazil,206.22,0.0011,0.10486,-0.024711,0.058012,brazil
319,Brazil,2014-01-31,737.863,-0.007358,5,0.223771,5.867838e+10,0.0,5.867838e+10,2.933919e+10,20.35,brazil,205.43,0.0010,0.10486,-0.024711,0.058012,brazil
